# RQ1 — effort-reduction curve (TESSERA, 0%–100% expert)

**Purpose.** Reads and plots the effort-reduction curve of RQ1
(`docs/research_questions.md#rq1--citizen-science-as-a-substitute-for-expert-data`):
citizen-science presences as a fixed base, expert polygons added on top
from 0% to 100% in a nested order, the expert-only model as a separate
reference point, and "100% + citizen science" as the additive end of the
curve. Every model is scored on the pooled test presences and on each
source separately.

**Design (2026-11-17, presence-only):** presence-background logistic
regression (`C=10`, chosen in `presence_only_method_selection.ipynb`),
leave-one-block-out over the 10 km blocks with at least 5 presences of
either source, sensitivity at 10% of predicted area as the primary
metric, ROC-AUC per fold as secondary. Reasoning:
[`thesis/notes.md`](../../thesis/notes.md#curva-presence-only).

This notebook trains nothing: the computation is
`experiments/effort_reduction_curve/run.py` (and `compare_buffer.py` for
the buffer sensitivity analysis). Run them before this notebook.


## 1. Load the saved run

The hashes of the input files recorded in `run.json` are compared with
the current files: if one differs, the tables below were computed from
other data and the script has to be run again.


In [ ]:
import json
import sys
import warnings
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src import manifest

OUT_DIR = ROOT / "results" / "effort_reduction_curve"
run = json.loads((OUT_DIR / "run.json").read_text())
assert not run["smoke_test"], "the saved run is a smoke test, not a result"
per_fold = pd.read_csv(OUT_DIR / "per_fold.csv", dtype={"condition": str})
summary = pd.read_csv(OUT_DIR / "summary.csv", dtype={"condition": str})
pooled_curve = pd.read_csv(OUT_DIR / "pooled_curve.csv", dtype={"condition": str})
paired = pd.read_csv(OUT_DIR / "paired_differences.csv")
test_sizes = pd.read_csv(OUT_DIR / "test_sizes.csv")
nominal = pd.read_csv(OUT_DIR / "nominal_levels.csv", dtype={"condition": str})
CONDITIONS = run["conditions"]            # "0" ... "100", "100% + citizen science"
PLUS_CS = CONDITIONS[-1]
SUBSETS = ["pooled", "expert only", "citizen science only"]
SENS = f"sensitivity_at_{int(run['primary_area_fraction'] * 100)}pct"

changed = [p for p, h in run["inputs_sha256"].items() if manifest.sha256_path(ROOT / p) != h]
if changed:
    warnings.warn(f"These inputs changed since the run of {run['run_date']}: {changed}. "
                  "Run experiments/effort_reduction_curve/run.py again.")
else:
    print(f"run of {run['run_date']} ({run['script']}): the input files are the current ones")
print(f"citizen-science presences: {run['n_cs_presences']:,} (matched {run['n_cs_matched']}/{run['n_cs_rows']})")
print(f"expert presence pixels: {run['n_expert_pixels']:,}, across {run['n_expert_polygons']:,} polygons "
      f"({run['n_expert_polygons_crossing_blocks']} cross a block border)")
print(f"background points: {run['n_background']:,} ({run['background']}); buffer: {run['buffer_m']:g} m")
print(f"{run['n_folds']} folds (blocks with >= {run['min_presences_per_fold']} presences of either source)")


## 2. Effort order and mixing levels

A fixed random order over the expert polygons (seed 0) decides which
polygons count as effort already spent at each level; the 20% set
contains the 10% set. `total_N` is nominal (the full dataset, not one
training fold): the citizen-science base plus the expert pixels of the
included polygons. Condition `100` is the expert-only model (no
citizen-science base); the additive end of the curve is
`100% + citizen science`.


In [ ]:
nominal

## 3. Folds and test sizes

Each fold holds out one block: its citizen-science presences, its expert
pixels (by polygon) and its background points. The same test set is used
at every level. Folds are unequal: one block holds 42% of the expert
test pixels, which is why the pooled numbers are reported next to the
per-fold ones.


In [ ]:
display(test_sizes)
print(f"{(test_sizes['n_test_expert_pixels'] > 0).sum()} folds with expert test pixels, "
      f"{(test_sizes['n_test_cs'] > 0).sum()} with citizen-science test presences, "
      f"{((test_sizes['n_test_expert_pixels'] > 0) & (test_sizes['n_test_cs'] > 0)).sum()} with both")
print(f"largest share of expert test pixels in one fold: "
      f"{test_sizes['n_test_expert_pixels'].max() / test_sizes['n_test_expert_pixels'].sum():.0%}")


## 4. Sensitivity at 10% of area and ROC-AUC, by level and source of the test presences

`pooled_*`: every held-out presence counted once, ranked against the
background of its own block. `per_fold_*`: mean and sd over folds
(blocks weighted equally). Chance level of the sensitivity is 0.10.


In [ ]:
def wide(df, value):
    w = df.pivot(index="test_presences", columns="condition", values=value)
    return w.loc[SUBSETS, CONDITIONS]

print("Pooled sensitivity at 10% of area:")
display(wide(summary, f"pooled_{SENS}").round(3))
print("Per-fold sensitivity at 10% of area, mean (sd):")
display((wide(summary, f"per_fold_{SENS}_mean").round(3).astype(str) + " ("
         + wide(summary, f"per_fold_{SENS}_sd").round(3).astype(str) + ")"))
print("Per-fold ROC-AUC against the block background, mean:")
display(wide(summary, "per_fold_roc_auc_mean").round(3))


## 5. Paired differences per fold

Each level is compared with 0% inside each fold (same test block); the
primary comparison of RQ1 is 90% minus 0% on the pooled test presences.
`pooled_diff` is the difference of the pooled values; the per-fold
columns give the mean, sd and the number of folds up, down and tied.


In [ ]:
key = ["90 minus 0", "50 minus 0", "10 minus 0", f"{PLUS_CS} minus 90", f"{PLUS_CS} minus 100"]
display(paired[paired["comparison"].isin(key)].set_index(["test_presences", "comparison"]).loc[SUBSETS].round(3))


## 6. Plot: sensitivity at 10% of area by level and test subset

One panel per source of the test presences, shared y-axis. The line is
the additive curve (citizen-science base at every level; the last point
is 100% of the polygons plus citizen science). The hollow marker at 100
is the expert-only model, a separate reference point. The band is ±1 sd
over folds of the per-fold value; the line is the pooled value.


In [ ]:
import matplotlib.pyplot as plt

LINE, REF, BAND = "#2a5d9f", "#2a5d9f", "#2a5d9f"
PANELS = [("pooled", "Pooled test presences"), ("expert only", "Expert presences only"),
          ("citizen science only", "Citizen-science presences only")]
levels_num = [int(c) for c in CONDITIONS if c != PLUS_CS and c != "100"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True)
for ax, (subset, title) in zip(axes, PANELS):
    s = summary[summary["test_presences"] == subset].set_index("condition")
    x = levels_num + [100]
    conds = [str(l) for l in levels_num] + [PLUS_CS]
    m = s.loc[conds, f"pooled_{SENS}"].values
    fm, fsd = s.loc[conds, f"per_fold_{SENS}_mean"].values, s.loc[conds, f"per_fold_{SENS}_sd"].values
    ax.fill_between(x, fm - fsd, fm + fsd, color=BAND, alpha=0.12, linewidth=0)
    ax.plot(x, m, color=LINE, marker="o", markersize=5, linewidth=2, label="citizen science + expert (pooled)")
    ax.plot([100], [s.loc["100", f"pooled_{SENS}"]], marker="o", markersize=9, markerfacecolor="white",
            markeredgecolor=REF, markeredgewidth=2, linestyle="none", label="expert only (reference)")
    ax.axhline(run["primary_area_fraction"], color="0.6", linewidth=1, linestyle=":", label="chance")
    ax.set_title(f"{title} (n = {int(s.loc['0', 'n_test_presences']):,})", fontsize=11)
    ax.set_xlabel("% of expert polygons added to the citizen-science base")
    ax.set_xticks(levels_num + [100])
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
axes[0].set_ylabel("sensitivity at 10% of predicted area")
axes[0].set_ylim(0, 1)
axes[0].legend(loc="lower right", frameon=False, fontsize=9)
fig.suptitle(f"RQ1 — effort-reduction curve, leave-one-block-out ({run['n_folds']} folds), TESSERA 2023",
             fontsize=12)
fig.tight_layout()
fig.savefig(OUT_DIR / "curve_by_source.png", dpi=150)
plt.show()


### 6b. Sensitivity-by-area curves

Pooled sensitivity at every area fraction from 1% to 50%, for 0%, 50%
and 90% of the polygons and the expert-only model. The primary number
is read at 10% (dotted line).


In [ ]:
SHOW = {"0": ("0% (citizen science only)", "#2a5d9f", "-"), "50": ("50%", "#2a5d9f", "--"),
        "90": ("90%", "#2a5d9f", ":"), "100": ("expert only", "#b5541c", "-")}
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True)
for ax, (subset, title) in zip(axes, PANELS):
    for cond, (label, color, ls) in SHOW.items():
        c = pooled_curve[(pooled_curve["test_presences"] == subset) & (pooled_curve["condition"] == cond)]
        ax.plot(c["area_fraction"], c["sensitivity"], color=color, linestyle=ls, linewidth=2, label=label)
    ax.plot([0, 0.5], [0, 0.5], color="0.6", linewidth=1, linestyle=":", label="chance")
    ax.axvline(run["primary_area_fraction"], color="0.8", linewidth=1)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("fraction of predicted area")
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
axes[0].set_ylabel("pooled sensitivity")
axes[0].legend(loc="lower right", frameon=False, fontsize=9)
fig.tight_layout()
plt.show()


## 7. Sensitivity to a buffer between training and test

Reads `results/effort_reduction_curve_buffer_comparison/` (written by
`compare_buffer.py` from the runs with `--buffer-m 300` and
`--buffer-m 1000`). Criterion, fixed before running: a comparison is
met if the absolute mean of the per-fold differences (with buffer minus
without) is smaller than their sd across folds; the buffer does not
change the reading of RQ1 only if all 8 comparisons are met. Reasoning:
[`thesis/notes.md`](../../thesis/notes.md#validação-espacial-no-lado-dos-especialistas-zona-tampão-além-dos-blocos-de-10-km).


In [ ]:
CMP_DIR = ROOT / "results" / "effort_reduction_curve_buffer_comparison"
if not (CMP_DIR / "run.json").exists():
    print("buffer comparison not run yet (experiments/effort_reduction_curve/compare_buffer.py)")
else:
    cmp_run = json.loads((CMP_DIR / "run.json").read_text())
    stale = [p for p, h in cmp_run["inputs_sha256"].items() if manifest.sha256_path(ROOT / p) != h]
    if stale:
        warnings.warn(f"buffer comparison older than its inputs: {stale}; run compare_buffer.py again")
    cmp = pd.read_csv(CMP_DIR / "paired_vs_no_buffer.csv")
    removed = pd.read_csv(CMP_DIR / "removed_from_training.csv")
    print("criterion met (sensitivity at 10%, 8 comparisons per buffer):", cmp_run["criterion_met_count"])
    print("share of training points removed by the buffer, mean over folds:")
    display(removed.assign(share=removed["n_removed"] / removed["n_train_before"])
            .groupby(["buffer_m", "source"])["share"].mean().unstack().round(3))
    sens = cmp[cmp["metric"] == "sensitivity_at_10pct"]
    display(sens.set_index(["buffer_m", "test_presences", "what"])
            [["no_buffer_pooled", "buffer_pooled", "diff_mean", "diff_sd", "folds_up", "folds_down", "criterion_met"]].round(3))


## 8. Reading the curve

Numbers from the tables above (one run, 2026-11-17); the full reading,
hypotheses and caveats are in
[`thesis/notes.md`](../../thesis/notes.md#curva-presence-only).

- **Pooled test presences (the primary comparison):** sensitivity at
  10% of area goes from 0.531 at 0% to 0.692 at 90% (pooled difference
  +0.161; per fold +0.181, sd 0.221, 18 of 26 folds up, 6 down, 2
  tied). The curve rises early and saturates: about half of the gain
  is there at 10% of the polygons, nearly all by 50%.
- **Expert presences only:** 0.637 → 0.933 (+0.296; 17 of 17 folds
  up). The expert-only model reaches 0.983.
- **Citizen-science presences only:** 0.411 → 0.417 (+0.006; 6 folds
  up, 7 down, 6 tied): adding expert polygons does not change how the
  citizen-science presences are ranked. The expert-only model is the
  worst here (0.381).
- **Where the gain comes from:** in the 9 folds whose test set has
  only citizen-science presences, 90% minus 0% averages −0.052; in the
  7 expert-only folds, +0.396; in the 10 folds with both, +0.241. The
  largest fold (42% of the expert test pixels) sits near the per-fold
  mean, so the pooled value is not driven by it.
- **Adding citizen science to all the expert polygons** changes
  nothing against 90% (−0.001) and, against the expert-only model,
  costs 0.048 on the expert presences and gains 0.032 on the
  citizen-science ones.
- **Buffer (section 7):** with 300 m or 1 km between training and
  test, sensitivity at 10% of area moves by at most 0.006 (mean paired
  difference per fold) and the 90% minus 0% difference does not
  change; the pre-written criterion is met in all 8 comparisons at
  both distances. The reading above does not depend on the buffer.
- **Caveats:** only 17 folds have expert test pixels; the per-fold sd
  is large; H₁ was written after the earlier (Random Forest) curve; the
  absolute level is not read, only the comparison between conditions.
